# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nID: {metadata.id}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the record sets present in the dataset as well as their field and column `@id`s.

Entities are always referenced using their `@id`.

In [ ]:
# List the available record sets, their fields, and columns with their `@id`s

record_sets = dataset.record_sets

print("Available record sets and their field/column @id's:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | dtype: {field.data_type}")
        if getattr(field, 'columns', None):
            print("      Columns:")
            for col in field.columns:
                print(f"        - {col.name} (@id: {col.id}) | dtype: {col.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all records from the main record set. Make sure to use the record set `@id` as the key identifier.

In [ ]:
# Gather all record set @ids for programmatic processing

record_sets_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from RecordSet @id: {record_set_id}")

# Display the column names from the primary record set (first one)
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id is not None:
    print(f"\nFields (columns) in RecordSet @id {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets present in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Fields are always referenced by their `@id`. Adjust the field `@id` values as needed from the overview section above.

In [ ]:
# Example: Select a numeric field (replace with an actual field @id found earlier)

# Use your exploration above to identify a numeric field (e.g., 'MW_kDa', 'Peptide_counts', etc.)
# For this example, we'll try commonly found IDs (MASS_SPECTROMETRY_RECORDSET_ID and MOLECULAR_WEIGHT_FIELD_ID SHOULD BE UPDATED)
numeric_field_id = None

# Try to programmatically detect the first float/integer column
import numpy as np
df = dataframes[main_record_set_id]
candidate_fields = df.select_dtypes(include=[np.number]).columns.tolist()
if candidate_fields:
    numeric_field_id = candidate_fields[0]
    print(f"Automatically selected numeric field (by @id): {numeric_field_id}")
else:
    print("No numeric fields automatically detected. Please manually enter the numeric field @id.")

# Example filtering: Only use if numeric_field_id is not None
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} rows")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records, showing top 5:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a key attribute if possible (e.g., 'accession' or categorical field)
    # Pick first object-type field (string/category)
    group_field = None
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c]):
            group_field = c
            break

    if group_field is not None and group_field != numeric_field_id:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field}, showing mean of {numeric_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field detected.")
else:
    print("Skipping EDA as no numeric field was identified.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the main numeric field (if detected)
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a categorical group field is available, plot boxplot
    if group_field is not None and group_field != numeric_field_id:
        top_n = filtered_df[group_field].value_counts().head(5).index.tolist()
        subset = filtered_df[filtered_df[group_field].isin(top_n)]
        plt.figure(figsize=(10,5))
        sns.boxplot(
            data=subset,
            x=group_field,
            y=numeric_field_id,
            palette="Set2"
        )
        plt.title(f"{numeric_field_id} grouped by {group_field} (Top 5 {group_field})")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded Croissant-compliant metadata and data from a life sciences dataset using `mlcroissant`.
- We reviewed available record sets, fields, and referenced all entities by their `@id` for reproducibility.
- We extracted and tabulated data using pandas, and performed EDA for numeric fields including filtering, normalization, aggregation and visualization.
- This notebook can be extended for more advanced analyses (e.g., downstream modeling, biological interpretation) using the field `@id` references above.